In [1]:
# from tensorflow import keras
# model = keras.models.load_model("models/CNN_Manakov_full.keras") 
# model.save("models/CNN_Manakov_full.h5", save_format="h5")

In [2]:
# Run these lines only the first time you run this notebook

# !pip install tensorflow_addons gdown shap -q
# !pip install git+https://github.com/katarinagresova/DeepExperiment

# !wget https://github.com/ML-Bioinfo-CEITEC/miRBind/raw/main/Models/miRBind.h5
# !wget https://raw.githubusercontent.com/ML-Bioinfo-CEITEC/miRBind/graphs/Datasets/evaluation_set_1_1_CLASH2013_paper.tsv

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import tensorflow

import json
import random
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import register_keras_serializable
# import tensorflow_addons as tfa
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from textwrap import wrap
from shap.plots.colors import red_transparent_blue

# from deepexperiment.alignment import Attrament
# from deepexperiment.visualization import plot_alignment, plot_miRNA_importance, plotbar_miRNA_importance
from utils.encoding import one_hot_encoding, one_hot_encoding_batch, binding_encoding
# from deepexperiment.interpret import DeepShap

from constants import HELA_TRANSFACTION_DATA, PSILAC_DATA, TARGETSCAN_COLUMN_TO_SEQUENCE

2025-07-02 14:15:47.933426: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-02 14:15:52.468996: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [5]:
import shap

# these layers are not supported, so the workaround presented on the github is to use passthrough
shap.explainers._deep.deep_tf.op_handlers["AddV2"] = shap.explainers._deep.deep_tf.passthrough 
shap.explainers._deep.deep_tf.op_handlers["FusedBatchNormV3"] = shap.explainers._deep.deep_tf.passthrough
shap.explainers._deep.deep_tf.op_handlers["LeakyRelu"] = shap.explainers._deep.deep_tf.passthrough

Params

In [6]:
DATASET_NAME = 'refseq_id.mirna_fc'
# DATASET_NAME = 'gene_name.psilac'
# DATASET_NAME = 'refseq_id.HeLa_transfection'

# ID_COLUMN = "refseq_mrna"
ID_COLUMN = 'RefSeq ID'

SAVE_EXPL_SCORES = True
# SAVE_EXPL_SCORES = False

In [7]:
TS_mirnas = list(TARGETSCAN_COLUMN_TO_SEQUENCE.items())
# mirna_index = 0
# mirna_index = 1
# mirna_index = 2
# mirna_index = 3
# mirna_index = 4
# mirna_index = 5
mirna_index = 6
mirna_name = TS_mirnas[mirna_index][0]
my_miRNA = TS_mirnas[mirna_index][1]

mirna_sequences = [my_miRNA]
mirna_sequences, mirna_name

(['AGCAGCATTGTACAGGGCTATGA'], 'hsa-miR-103a-3p')

In [8]:
TS_mirnas

[('hsa-miR-16-5p', 'TAGCAGCACGTAAATATTGGCG'),
 ('hsa-miR-106b-5p', 'TAAAGTGCTGACAGTGCAGAT'),
 ('hsa-miR-200a-3p', 'TAACACTGTCTGGTAACGATGT'),
 ('hsa-miR-200b-3p', 'TAATACTGCCTGGTAATGATGA'),
 ('hsa-miR-215-5p', 'ATGACCTATGAATTGACAGAC'),
 ('hsa-let-7c-5p', 'TGAGGTAGTAGGTTGTATGGTT'),
 ('hsa-miR-103a-3p', 'AGCAGCATTGTACAGGGCTATGA')]

In [9]:
# SEQUENCE_SOURCE_PATH = "data/3utr/3utr.sequences.pkl"
# SEQUENCE_SOURCE_PATH = 'data/3utr/3utr.sequences.refseq_id.mirna_fc.pkl'
# SEQUENCE_SOURCE_PATH = f'data/3utr/3utr.sequences.{DATASET_NAME}.pkl'
# SEQUENCE_SOURCE_PATH = f'data/processed/GRCh37.p13 hg19/3utr.sequences.{DATASET_NAME}.pkl'

SEQUENCE_SOURCE_PATH = f'data/processed/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.pkl'


sequence_source_df = pd.read_pickle(SEQUENCE_SOURCE_PATH)

In [10]:
sequence_source_df.columns

Index(['RefSeq ID', 'Gene symbol', 'hsa-miR-16-5p', 'hsa-miR-106b-5p',
       'hsa-miR-200a-3p', 'hsa-miR-200b-3p', 'hsa-miR-215-5p', 'hsa-let-7c-5p',
       'hsa-miR-103a-3p', 'knownGene.name', 'knownGene.chrom', 'kgAlias.kgID',
       'kgAlias.alias', 'kgXref.kgID', 'kgXref.mRNA', 'kgXref.geneSymbol',
       'kgXref.refseq', 'knownToEnsembl.name', 'knownToEnsembl.value',
       'knownToRefSeq.name', 'knownToRefSeq.value', 'ID_versioned', 'Name',
       'Description', 'sequence', 'Chromosome', 'Start', 'End', 'Strand',
       'Transcript ID', 'Representative transcript?', 'ensembl_id_no_version',
       'sequence_origin', 'utr3_length'],
      dtype='object')

In [11]:
# SAVE_EXPLAINABILITY_SCORES_PATH = "explainability_scores_{}.json".format(mirna_name)
# SAVE_EXPLAINABILITY_SCORES_PATH = "data/3utr.explainability_scores_{}.json".format(mirna_name)
# SAVE_EXPLAINABILITY_SCORES_PATH = f"data/3utr.sequences.refseq_id.mirna_fc.explainability_scores_{mirna_name}.json"
# SAVE_EXPLAINABILITY_SCORES_PATH = f"data/3utr.sequences.refseq_id.mirna_fc.explainability_scores_{mirna_name}.refseq_id.json"
# SAVE_EXPLAINABILITY_SCORES_PATH = f"debug/3utr.sequences.refseq_id.mirna_fc.explainability_scores_{mirna_name}.refseq_id.json"
# SAVE_EXPLAINABILITY_SCORES_PATH = f"data/3utr.sequences.{DATASET_NAME}.explainability_scores_{mirna_name}.json"

# SAVE_EXPLAINABILITY_SCORES_PATH = f"data/scanned/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.explainability_scores_{mirna_name}.json"

# SAVE_EXPLAINABILITY_SCORES_PATH = f"data/scanned/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.explainability_scores_{mirna_name}.json"

# SAVE_SCANNING_ERRORS_PATH = f"data/scanned/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.explainability_scores_{mirna_name}.scanning_errors.txt"

In [12]:
# CHANGED naming of explainability_scores_ to gradient_scores_ or grad_exp_mirbench_
# SCAN_MODE_EXTENSION = 'explainability_scores_'
# SCAN_MODE_EXTENSION = 'gradient_scores_'
SCAN_MODE_EXTENSION = 'grad_exp_mirbench_'

SAVE_EXPLAINABILITY_SCORES_PATH = f"data/scanned/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.{SCAN_MODE_EXTENSION}{mirna_name}.json"

SAVE_SCANNING_ERRORS_PATH = f"data/scanned/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.{SCAN_MODE_EXTENSION}{mirna_name}.scanning_errors.txt"

In [13]:
PREDICTION_THRESHOLD = 0
random.seed(42)

### Load and preprocess the transcript data

In [14]:
sequence_source_df.shape[0], sequence_source_df[ID_COLUMN].nunique(), sequence_source_df[[ID_COLUMN]].dropna().shape[0]

(8288, 8288, 8288)

In [15]:
gene_symbol_to_seq = sequence_source_df[[ID_COLUMN, "sequence"]].set_index(ID_COLUMN).to_dict()['sequence']

### Get our deep learning model and miRNA data

In [16]:
"""# Loading model and the data"""

# model = keras.models.load_model("models/miRBind.h5")   # Old model from miRBind trained on Ago1 data
model = keras.models.load_model("models/CNN_Manakov_full.keras")
# model = keras.models.load_model("models/CNN_Manakov_full.h5") 
# model = keras.models.load_model("models/CNN_Manakov_full.h5", compile=False)
# miRNA/miRNA/models/model_miRNA.h5
# model = keras.models.load_model("models/model_miRNA.h5")   # from Vasek's paper https://github.com/ML-Bioinfo-CEITEC/HybriDetector/blob/main/ML/Models/model_miRNA.h5
# model.summary()

2025-07-02 14:16:07.068479: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 18071 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe MIG 2g.20gb, pci bus id: 0000:61:00.0, compute capability: 8.0


In [17]:
# samples = pd.read_csv('evaluation_set_1_1_CLASH2013_paper.tsv', sep='\t')
samples = pd.read_csv("eclip/AGO2_eCLIP_Manakov2022_test.tsv", sep="\t")
samples.head()

,gene,noncodingRNA,noncodingRNA_name,noncodingRNA_fam,feature,label,chr,start,end,strand,gene_cluster_ID
0,CCCCAAATGTGGGAAACTTGACTGTATAATTTGTGGCAGTGGTAGA...,CTATACAATCTACTGTCTTTC,hsa-let-7a-3p,let-7,exon,1,1,146052098.0,146052147,-,34633
1,TGTACTGTGAGATTGCCCGGTACAGCAGCAGTTGTATTCTTTATTA...,CTATACAATCTACTGTCTTTC,hsa-let-7a-3p,let-7,five_prime_utr,1,1,114749943.0,114749992,-,9074
2,GATGTACAAGAAATAAATTAGGAGAGATTACTTTGTATTGTACTGC...,CTATACAATCTACTGTCTTTC,hsa-let-7a-3p,let-7,five_prime_utr,1,1,178746394.0,178746443,-,213724
3,AAATGTAAATGAAAAGAACTACTGTATATTAAAAGTTGGTTTGAAC...,CTATACAATCTACTGTCTTTC,hsa-let-7a-3p,let-7,three_prime_utr,1,1,11107324.0,11107373,-,473417
4,GGAGTAGAAAAATAGAATGGGAACATGTATAATCTTGTAATTCATA...,CTATACAATCTACTGTCTTTC,hsa-let-7a-3p,let-7,three_prime_utr,1,1,193087121.0,193087170,+,504839


In [18]:
samples.noncodingRNA.map(len).describe()

count    327129.000000
mean         22.379083
std           0.716805
min          16.000000
25%          22.000000
50%          22.000000
75%          23.000000
max          28.000000
Name: noncodingRNA, dtype: float64


### Use and evaluate the model

In [19]:
# n = 50 is number of samples used as a background image to compare the input with during the shap method
rand_samples = samples.sample(n=50, replace=False, random_state=42).reset_index(drop=True)
background, _ = binding_encoding(rand_samples)
# background, _ = one_hot_encoding_batch(rand_samples, mirna_column='noncodingRNA')


# deepShap = DeepShap(model, background)
explainer = shap.GradientExplainer(model, background)

#### #DEBUG Set `threshold_len` to filter out longer genes

In [20]:
threshold_len = -1 # -1
threshold_len_high = -1 # -1

# if threshold_len > 0:
#     count = 0
#     for key,value in gene_symbol_to_seq.items():
#         if(len(value) < threshold_len):
#             count += 1
#     count

if threshold_len > 0:
    count = 0
    for row in sequence_source_df[["ensembl_gene_id", "sequence"]].itertuples():
        if(len(row.sequence) < threshold_len):
            count += 1
    print(count)

In [21]:
# if threshold_len > 0:
#     count_too_long = 0
#     count_kept = 0
#     for key,value in gene_symbol_to_seq.items():
#         if(len(value) > threshold_len):
#             gene_symbol_to_seq[key] = []
#             count_too_long += 1
#         else:
#             count_kept += 1
#     print('count_too_long, len > ', threshold_len, " : ", count_too_long)
#     print('count_kept ', count_kept)


if threshold_len > 0:
    gene_symbol_to_seq = {}
    count_too_long = 0
    count_kept = 0
    for row in sequence_source_df[["ensembl_gene_id", "sequence"]].itertuples():
        # if(len(row.sequence) < threshold_len or len(row.sequence) > threshold_len_high):
            # count_too_long += 1
        # else:
        #     gene_symbol_to_seq[row.ensembl_gene_id] = row.sequence
        #     count_kept += 1
        if(len(row.sequence) < threshold_len): 
            gene_symbol_to_seq[row.ensembl_gene_id] = row.sequence
            count_kept += 1
    print('count_too_long, len > ', threshold_len, " : ", count_too_long)
    print('count_kept ', count_kept)

### Scan the transcript

In [22]:
def process_shap_attribution(shap_values, preds, shap_indices, pred_indices, 
                       gene_length, sequence_length, sample_index, 
                       multiply_by_predictions=True):
    if multiply_by_predictions:
        # Normalize SHAP values by multiplying with predictions
        normalized_shap = shap_values[0][sample_index, :, :, 0] * preds[shap_indices[sample_index]][0]
    else:
        # Use raw SHAP values without prediction multiplication
        normalized_shap = shap_values[0][sample_index, :, :, 0]

    # Add zero padding before the sequence
    newrows_before = np.zeros((pred_indices[sample_index], normalized_shap.shape[1]))
    normalized_shap = np.vstack([newrows_before, normalized_shap])

    # Add zero padding after the sequence
    newrows_after = np.zeros((gene_length - pred_indices[sample_index] - sequence_length, 
                             normalized_shap.shape[1]))
    normalized_shap = np.vstack([normalized_shap, newrows_after])
    return normalized_shap

In [23]:
def score_sequence_attribution_minimal_with_preds(
    gene, input_miRNA, model, gene_id, mirna_id, draw_plot=False, step=10, length=50, prediction_threshold=0.5,
):
    # QUICK FIX - miRBind takes only 20 long miRNA
    miRNA = input_miRNA[0:20]
    miRNAs = []
    genes = []
    counts = np.zeros(len(gene))
    for i in range(0, len(gene) - length + 1, step):
        start = max(i, 0)
        end = min(i+length, len(gene))
        miRNAs.append(miRNA)
        genes.append(gene[start:end])
        counts[start:end] += 1
    labels = np.zeros(len(genes))
    df = pd.DataFrame(
        {'noncodingRNA': miRNAs,
         'gene': genes,
         'label': labels
        })
    # try:
    data, _ = one_hot_encoding_batch(df, tensor_dim=(50, 20, 1), mirna_column='noncodingRNA')
        # data, _ = binding_encoding(df, tensor_dim=(50, 20, 1))
    # except Exception as err:
        # print(type(err), err)
        # print(df)
        # pass
    preds = model(data)
    
    # Create extended predictions array to match gene length
    preds_extended = np.zeros(len(gene))
    preds_counts = np.zeros(len(gene))  # To handle overlapping windows
    
    counter = 0
    for i in range(0, len(gene) - length + 1, step):
        prediction = preds[counter][0]  # Access the first (and only) element
        start = max(i, 0)
        end = min(i+length, len(gene))
        
        # Add prediction to all positions covered by this window
        preds_extended[start:end] += prediction
        preds_counts[start:end] += 1
        counter += 1
    
    # Average predictions for overlapping regions
    preds_counts[preds_counts == 0] = 1  # Avoid division by zero
    preds_extended = preds_extended / preds_counts
    
    attribution_raw = np.zeros((len(gene), len(miRNA)))
    attribution_multiplied_by_preds = np.zeros((len(gene), len(miRNA)))
    shap_indices = []
    pred_indices = []
    preds_indices = []
    shap_data = []
    counter = 0
    
    for i in range(0, len(gene) - length + 1, step):
        # Access the single prediction value directly
        prediction = preds[counter][0]  # Access the first (and only) element
            
        # NOTE I would assume here will be an else which puts zeroes to the index, but I quess that is not necessary since we initialize zero array and then only input non zero values to specific places?
        if prediction > prediction_threshold:
            shap_indices.append(counter)
            pred_indices.append(i)
            shap_data.append(data[counter])
        counter += 1
    
    # TODO quick fix: if at this point the 'shap_data' is empty array [], it means no prediction was > 0.5, therefore (for now) let's predict 0 binding affinity  
    if len(shap_data) == 0:
        return []
    shap_data = np.stack(shap_data)
    
    # neg_shap, pos_shap = deepShap(shap_data)
    shap_values = explainer.shap_values(data)
    # print(np.array(shap_values).shape)

    for i in range(0, len(shap_indices)):
        sample_attribution = process_shap_attribution(
            shap_values, preds, shap_indices, pred_indices, 
            gene_length=len(gene), sequence_length=length, sample_index=i, multiply_by_predictions=False
        )
        attribution_raw += sample_attribution
        
        sample_attribution = process_shap_attribution(
            shap_values, preds, shap_indices, pred_indices, 
            gene_length=len(gene), sequence_length=length, sample_index=i, multiply_by_predictions=True
        )
        attribution_multiplied_by_preds += sample_attribution
        
    attribution_raw = attribution_raw.T.max(axis=0)
    attribution_multiplied_by_preds = attribution_multiplied_by_preds.T.max(axis=0)
 
    counts[counts == 0] = 1 # because when stepping transcript, its len might not be dividable by step and leave a few 0s at the end of counts
    scores_raw = attribution_raw / np.array(counts)
    scores_multiplied_by_preds = attribution_multiplied_by_preds / np.array(counts)
    
    if draw_plot:
        random_plot_number = random.randint(0, 1000)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), num=random_plot_number)

        # Top subplot - scores
        ax1.plot(scores_raw, label='scores_raw')
        ax1.plot(scores_multiplied_by_preds, label='scores_multiplied_by_preds')
        ax1.legend()
        ax1.set_title('Attribution scores')
        ax1.grid(True, alpha=0.3)

        # Bottom subplot - predictions
        ax2.plot(preds_extended, label='predictions_extended', color='orange')
        ax2.legend()
        ax2.set_title('Predictions')
        ax2.grid(True, alpha=0.3)

        # Overall title and layout
        title = f'{gene_id}_{mirna_id}_plot-id-{random_plot_number}'
        fig.suptitle(title)
        plt.tight_layout()

        plt.savefig('gradient_plots/' + title + '.png', dpi=300, bbox_inches='tight')
        plt.close()
        # plt.show()
        
    # print(np.array(scores_raw).shape)
    # print(np.array(scores_multiplied_by_preds).shape)
    # print(np.array(preds_extended).shape)  
    
    # Return the extended predictions along with the other results
    return scores_raw, scores_multiplied_by_preds, preds_extended, preds

In [24]:
# def score_sequence_attribution_minimal(gene, input_miRNA, model, draw_plot=False, step=10, length=50, prediction_threshold=0.5):
#     # QUICK FIX - miRBind takes only 20 long miRNA
#     miRNA = input_miRNA[0:20]
#     miRNAs = []
#     genes = []
#     counts = np.zeros(len(gene))
#     for i in range(0, len(gene) - length + 1, step):
#         start = max(i, 0)
#         end = min(i+length, len(gene))
#         miRNAs.append(miRNA)
#         genes.append(gene[start:end])
#         counts[start:end] += 1
#     labels = np.zeros(len(genes))
#     df = pd.DataFrame(
#         {'noncodingRNA': miRNAs,
#          'gene': genes,
#          'label': labels
#         })
#     # try:
#     data, _ = one_hot_encoding_batch(df, tensor_dim=(50, 20, 1), mirna_column='noncodingRNA')
#         # data, _ = binding_encoding(df, tensor_dim=(50, 20, 1))
#     # except Exception as err:
#         # print(type(err), err)
#         # print(df)
#         # pass
#     preds = model(data)
    
#     attribution_raw = np.zeros((len(gene), len(miRNA)))
#     attribution_multiplied_by_preds = np.zeros((len(gene), len(miRNA)))
#     shap_indices = []
#     pred_indices = []
#     preds_indices = []
#     shap_data = []
#     counter = 0
    
#     for i in range(0, len(gene) - length + 1, step):
#         # Access the single prediction value directly
#         prediction_score = preds[counter][0]  # Access the first (and only) element
            
#         # NOTE I would assume here will be an else which puts zeroes to the index, but I quess that is not necessary since we initialize zero array and then only input non zero values to specific places?
#         if prediction_score > prediction_threshold:
#             shap_indices.append(counter)
#             pred_indices.append(i)
#             shap_data.append(data[counter])
#         counter += 1
    
#     # TODO quick fix: if at this point the 'shap_data' is empty array [], it means no prediction was > 0.5, therefore (for now) let's predict 0 binding affinity  
#     if len(shap_data) == 0:
#         return []
#     shap_data = np.stack(shap_data)
    
#     # neg_shap, pos_shap = deepShap(shap_data)
#     shap_values = explainer.shap_values(data)
#     print(np.array(shap_values).shape)

#     for i in range(0, len(shap_indices)):
#         # # Use the single output value directly - no need to index with [1]
#         # # normalized_shap = pos_shap[i,:,:,0] * preds[shap_indices[i]][0]
#         # normalized_shap = shap_values[0][i,:,:,0] * preds[shap_indices[i]][0]
#         # # CHANGE what if we don't multiply by preds
#         # # normalized_shap = shap_values[0][i,:,:,0]
#         # newrows = np.zeros((pred_indices[i], normalized_shap.shape[1]))
#         # normalized_shap = np.vstack([newrows, normalized_shap])
#         # newrows = np.zeros((len(gene) - pred_indices[i] - length, normalized_shap.shape[1]))
#         # normalized_shap = np.vstack([normalized_shap, newrows])
#         # attribution += normalized_shap
#         sample_attribution = process_shap_attribution(
#             shap_values, preds, shap_indices, pred_indices, 
#             gene_length=len(gene), sequence_length=length, sample_index=i, multiply_by_predictions=False
#         )
#         attribution_raw += sample_attribution
        
#         sample_attribution = process_shap_attribution(
#             shap_values, preds, shap_indices, pred_indices, 
#             gene_length=len(gene), sequence_length=length, sample_index=i, multiply_by_predictions=True
#         )
#         attribution_multiplied_by_preds += sample_attribution
        
#     attribution_raw = attribution_raw.T.max(axis=0)
#     attribution_multiplied_by_preds = attribution_multiplied_by_preds.T.max(axis=0)
 
#     counts[counts == 0] = 1 # because when stepping transcript, its len might not be dividable by step and leave a few 0s at the end of counts
#     normalized_scores_raw = attribution_raw / np.array(counts)
#     normalized_scores_multiplied_by_preds = attribution_multiplied_by_preds / np.array(counts)
    
#     if draw_plot:
#         random_plot_number = random.randint(0, 1000)
#         plt.figure(num = random_plot_number)
#         plt.plot(normalized_scores_raw, label='normalized_scores_raw')
#         plt.plot(normalized_scores_multiplied_by_preds, label='normalized_scores_multiplied_by_preds')
#         # plt.plot(preds, label='preds')
#         plt.legend()
#         title = f'normalized_by_pred.gradient_scores_plot_{random_plot_number}'
#         # title = f'without_multiplication_by_pred.gradient_scores_plot_{random_plot_number}'
#         plt.title(title)
#         plt.savefig('gradient_plots/' + title + '.png', dpi=300, bbox_inches='tight')  # Save the plot as PNG with good resolution
#         plt.show()
        
#     print(np.array(normalized_scores_raw).shape)
#     print(np.array(normalized_scores_multiplied_by_preds).shape)
#     # return normalized_scores
#     # CHANGE what if we don't multiply by preds
#     return normalized_scores_raw, normalized_scores_multiplied_by_preds, preds

In [25]:
# def score_with_gradients(gene, input_miRNA, model, draw_plot=False, step=10, length=50, prediction_threshold=0.5):
#     # Similar preprocessing as before
#     miRNA = input_miRNA[0:20]
#     miRNAs = []
#     genes = []
#     counts = np.zeros(len(gene))
#     for i in range(0, len(gene) - length + 1, step):
#         start = max(i, 0)
#         end = min(i+length, len(gene))
#         miRNAs.append(miRNA)
#         genes.append(gene[start:end])
#         counts[start:end] += 1
        
#     labels = np.zeros(len(genes))
#     df = pd.DataFrame({
#         'miRNA': miRNAs,
#         'gene': genes,
#         'label': labels
#     })
    
#     # data, _ = one_hot_encoding_batch(df, tensor_dim=(50, 20, 1))
#     data, _ = binding_encoding(df, tensor_dim=(50, 20, 1))
    
#     # Get predictions
#     preds = model(data)
    
#     # Initialize attribution
#     attribution = np.zeros(len(gene))
    
#     # Use TF's gradient tape for significant predictions
#     for idx, i in enumerate(range(0, len(gene) - length + 1, step)):
#         if preds[idx][0] > prediction_threshold:
#             # Compute gradients for this input
#             with tf.GradientTape() as tape:
#                 x = tf.convert_to_tensor(data[idx:idx+1], dtype=tf.float32)
#                 tape.watch(x)
#                 y = model(x)
            
#             # Get gradients
#             grads = tape.gradient(y, x)[0]
            
#             # Sum absolute gradients for feature importance
#             # Since we want importance per position in the gene, sum over miRNA dimension
#             importance = np.abs(grads.numpy()).sum(axis=2).sum(axis=1).squeeze()
            
#             # Map importance back to gene positions
#             start = max(i, 0)
#             end = min(i+length, len(gene))
            
#             # Now we can safely add the importance scores
#             # If window size differs from importance length, trim or pad
#             window_size = end - start
#             if window_size <= len(importance):
#                 # Use only the needed part of importance
#                 attribution[start:end] += importance[:window_size] * preds[idx][0].numpy()
#             else:
#                 # Pad with zeros if needed
#                 padded_importance = np.pad(importance, (0, window_size - len(importance)))
#                 attribution[start:end] += padded_importance * preds[idx][0].numpy()
    
#     # Normalize
#     counts[counts == 0] = 1
#     normalized_scores = attribution / counts
    
#     if draw_plot:
#         random_plot_number = random.randint(0, 1000)
#         plt.figure(num=random_plot_number)
#         plt.plot(normalized_scores)
#         plt.title('Gradient-based scores')
#         plt.savefig(f'gradient_plots/gradient_scores_plot_{random_plot_number}.png', dpi=300, bbox_inches='tight')  # Save the plot as PNG with good resolution
#         plt.show()
    
#     return normalized_scores

In [26]:
len(gene_symbol_to_seq.items())

8288

In [ ]:
import time

# both array and dictionary to collect results for now, will see which one is more handy
score_table = []
miRNA_to_gene_score = {}
explain_errors = []

start = time.time()

i = 0
for miRNA in mirna_sequences[:1]:
    if miRNA not in miRNA_to_gene_score:
        miRNA_to_gene_score[miRNA] = []
    for gene_symbol, gene_sequence in gene_symbol_to_seq.items():
        if not isinstance(gene_sequence, float) and len(gene_sequence) > 0:
            try:
                # score = score_with_gradients(
                # score = score_sequence_attribution_minimal(
                #     str(gene_sequence), 
                #     miRNA, 
                #     model, 
                #     # draw_plot=False, 
                #     draw_plot=True, 
                #     step=10, 
                #     length=50, 
                #     prediction_threshold=PREDICTION_THRESHOLD
                # )
                score_raw, score_mult_by_preds, preds_extended, preds = score_sequence_attribution_minimal_with_preds(
                    str(gene_sequence), 
                    miRNA, 
                    model, 
                    # draw_plot=False, 
                    draw_plot=True, 
                    step=10, 
                    length=50, 
                    prediction_threshold=PREDICTION_THRESHOLD,
                    gene_id=gene_symbol,
                    mirna_id=mirna_name,
                )
                # score_table.append([miRNA, gene_symbol, score])
                score_table.append([miRNA, gene_symbol, score_raw, score_mult_by_preds, preds_extended, preds])
                # miRNA_to_gene_score[miRNA].append([gene_symbol, score])
                miRNA_to_gene_score[miRNA].append([gene_symbol, score_raw, score_mult_by_preds, preds_extended, preds])
            except (AssertionError, ValueError) as e:
                # print(miRNA, gene_symbol, e)
                print(e)
                explain_errors.append([mirna_name, miRNA, gene_symbol, str(e)])
                
        else:
            explain_errors.append(gene_symbol)

        i+=1
        if i % 500 == 0:
        # if i % 100 == 0:
            print(gene_symbol, " |",i , "| " , end =" ")
        # if i > 10:
        #     break
    # break
        
            
end = time.time()

2025-07-02 14:16:10.188148: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8800
2025-07-02 14:16:10.515169: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
`tf.keras.backend.set_learning_phase` is deprecated and will be removed after 2020-10-11. To update it, simply pass a True/False value to the `training` argument of the `__call__` method of your layer or model.


not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4, got 0)
not enough values to unpack (expected 4,

In [ ]:
print(f'{round(end - start, 2)} seconds, {round((end - start) / 3600, 2)} hours')

In [ ]:
len(gene_symbol_to_seq.items())

In [ ]:
print(len(explain_errors))
explain_errors[:5]

In [ ]:
empties = 0
for gene_n_score in miRNA_to_gene_score[my_miRNA]:
    if len(gene_n_score[1]) == 0:
        empties += 1
print(empties, ' / ', len(miRNA_to_gene_score[my_miRNA]))

#### Save the explainability scoring to a file

In [ ]:
print(type(miRNA_to_gene_score), type(miRNA_to_gene_score[my_miRNA]))

In [ ]:
SAVE_EXPLAINABILITY_SCORES_PATH

In [ ]:
miRNA_to_gene_score.keys()

In [ ]:
# if SAVE_EXPL_SCORES:
#     with open(SAVE_EXPLAINABILITY_SCORES_PATH, 'w') as file:
#         data_to_save = {}
#         for key in miRNA_to_gene_score.keys():
#             # data_to_save[key] = [[sub_key, list(sub_val)] for sub_key, sub_val in miRNA_to_gene_score[key]]
#             data_to_save[key] = [[gene_symbol, list(score_raw), list(score_mult_by_preds), list(preds_extended), list(preds)] for gene_symbol, score_raw, score_mult_by_preds, preds_extended, preds in miRNA_to_gene_score[key]]
            
#         json.dump(data_to_save, file, indent=4)

In [ ]:
if SAVE_EXPL_SCORES:
    with open(SAVE_EXPLAINABILITY_SCORES_PATH, 'w') as file:
        data_to_save = {}
        for key in miRNA_to_gene_score.keys():
            converted_data = []
            for gene_symbol, score_raw, score_mult_by_preds, preds_extended, preds in miRNA_to_gene_score[key]:
                # Convert tensors to numpy arrays, then to lists
                converted_data.append([
                    gene_symbol,
                    score_raw.numpy().tolist() if hasattr(score_raw, 'numpy') else list(score_raw),
                    score_mult_by_preds.numpy().tolist() if hasattr(score_mult_by_preds, 'numpy') else list(score_mult_by_preds),
                    preds_extended.numpy().tolist() if hasattr(preds_extended, 'numpy') else list(preds_extended),
                    preds.numpy().tolist() if hasattr(preds, 'numpy') else list(preds)
                ])
            data_to_save[key] = converted_data
            
        json.dump(data_to_save, file, indent=4)

In [ ]:
data_to_save.keys()

In [ ]:
if SAVE_EXPL_SCORES:
    with open(SAVE_SCANNING_ERRORS_PATH, 'w') as filehandle:
        json.dump(explain_errors, filehandle)

In [ ]:
SAVE_SCANNING_ERRORS_PATH